In [1]:
# Import dependencies
import gc
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import seaborn as sns
import plotly.express as px

import matplotlib.pyplot as plt
import squarify
import matplotlib.cm as cm

import plotly.io as pio

In [2]:
# Display settings for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)  # Display only the first 10 rows in output

In [31]:
# df_leads = None  # Clear the DataFrame
# df_opps = None  # Clear the DataFrame
# df_leads2 = None  # Clear the DataFrame
# df_leads2 = None  # Clear the DataFrame
# import gc
# gc.collect()  # Force garbage collection to remove old references

In [3]:
# Load the Salesforce report exports from Excel files
duplicate_df = pd.read_excel('CustomerDataForCreditTeam.xlsx')  # Read default sheet for opportunities

In [7]:
# read all column names and show all below
print(duplicate_df.columns.tolist())

['CUST.NUM', 'CUST.DESC', 'CUST.NAME.MM', 'ADDRESS1', 'ADDRESS2', 'ADDRESS3', 'CITY', 'STATE', 'ZIP', 'COUNTRY', 'PHONE', 'FAX', 'EMAIL.ADDR', 'SLSM.NUM', 'SLSM.DESC', 'SLSM2.NUM', 'SLSM2.DESC', 'NAT.ACCT.MGR.AFF', 'NAM.SW', 'AGENCY', 'AGENCY.DESC', 'AGENT', 'AGENT.DESC', 'SUB.GROUP.MM', 'CUST.CLASS.SWX', 'S.BUY.COOP.SWX', 'COOP.NUM', 'COOP.DESC', 'COOP.DATE', 'WHSE.NUM', 'SHIP.WHSE.NUM', 'TERR.NUM', 'TERR.DESC', 'TYPE', 'TYPE.DESC', 'CRED.LIMIT', 'CRED.CODE', 'CRED.CODE.RSN', 'CRED.MGR.ID', 'MAST.ACCT.FLG', 'MASTER.ACCT', 'MAST.ACCT.INV', 'CURRENCY', 'STMT.CODE', 'FAX.STM.FLG', 'COLLECT.TYPE', 'AR.EMAIL.ADDR', 'DETAIL.AR', 'AR.TERM.NUM', 'PROHIBIT.SLS', 'ACCEPT.BO', 'ACCEPT.SUBS', 'BEST.PRICE', 'PRC.CODE', 'SHIP.VIA.NUM', 'SHIP.VIA.DESC', 'FRT.NUM', 'FRT.DESC', 'CONS.BILL.ACCT', 'SOURCE.NUM', 'SOURCE.DESC', 'MIN.INV.AMT', 'MIN.INV.AMT.SW', 'LANG.CODE', 'START.DATE', 'SD.YAMA.MM', 'LAST.UPD.DATE', 'PRC.PCK.TICK', 'PROHIBIT.SLS.1', 'ACCEPT.BO.1', 'ACCPT.BO.JET', 'FAX.INV.FLG', 'FAX.ACK.

In [30]:
#!pip install rapidfuzz

In [10]:
from rapidfuzz import fuzz
from rapidfuzz import process

In [ ]:
# Define a threshold for fuzzy matching
similarity_threshold = 85  

# Function to find fuzzy duplicates with different AFF.SMC.FLG
def find_fuzzy_duplicates(df, name_col, flag_col, threshold):
    # Creating an empty DataFrame to store duplicates
    duplicates = pd.DataFrame()
    checked_names = set()

    # Iterate through unique customer names
    for name in df[name_col].unique():
        # Skip already checked names
        if name in checked_names:
            continue
        
        # Find similar names using fuzzy matching
        similar_names = process.extract(
            name, df[name_col].unique(), scorer=fuzz.ratio, limit=None
        )
        
        # Filter similar names above the threshold
        similar_names = [n[0] for n in similar_names if n[1] >= threshold]
        
        # Mark these names as checked
        checked_names.update(similar_names)
        
        # Filter rows matching these similar names
        group = df[df[name_col].isin(similar_names)]
        
        # Check if there are different AFF.SMC.FLG values
        if group[flag_col].nunique() > 1:
            duplicates = pd.concat([duplicates, group])

    return duplicates

# Using the function on DataFrame
result = find_fuzzy_duplicates(
    duplicate_df, name_col="CUST.DESC", flag_col="AFF.SMC.FLG", threshold=similarity_threshold
)

# Display or save the result
result


In [12]:
# Save the result DataFrame to an Excel file
result.to_excel("fuzzy_duplicates_1.xlsx", index=False)


In [13]:
# Define a function to find fuzzy duplicates with additional ADDRESS1 consideration
def find_fuzzy_duplicates_with_address(df, name_col, address_col, flag_col, name_threshold, address_threshold):
    duplicates = pd.DataFrame()
    checked_pairs = set()  # Yargi - to keep track of checked (name, address) pairs

    for idx, row in df.iterrows():
        name, address = row[name_col], row[address_col]

        # Skip already checked name-address pairs
        if (name, address) in checked_pairs:
            continue

        # Find similar names and addresses
        similar_names = process.extract(name, df[name_col].unique(), scorer=fuzz.ratio, limit=None)
        similar_addresses = process.extract(address, df[address_col].unique(), scorer=fuzz.partial_ratio, limit=None)

        # Filter based on thresholds
        similar_names = [n[0] for n in similar_names if n[1] >= name_threshold]
        similar_addresses = [a[0] for a in similar_addresses if a[1] >= address_threshold]

        # Mark these name-address pairs as checked
        for sim_name in similar_names:
            for sim_address in similar_addresses:
                checked_pairs.add((sim_name, sim_address))

        # Filter rows matching similar names and addresses
        group = df[(df[name_col].isin(similar_names)) & (df[address_col].isin(similar_addresses))]

        # Check if there are different AFF.SMC.FLG values
        if group[flag_col].nunique() > 1:
            duplicates = pd.concat([duplicates, group])

    return duplicates

# Define thresholds
name_similarity_threshold = 85
address_similarity_threshold = 80 # Yargi - Lower threshold for partial matching

# Using the function on DataFrame
result_with_address = find_fuzzy_duplicates_with_address(
    duplicate_df, name_col="CUST.DESC", address_col="ADDRESS1", flag_col="AFF.SMC.FLG",
    name_threshold=name_similarity_threshold, address_threshold=address_similarity_threshold
)

# Saving the result to an Excel file
result_with_address.to_excel("fuzzy_duplicates_2.xlsx", index=False)


In [ ]:
# Ensure customer names are sorted alphabetically on the Y-axis
bar_data_sorted = bar_data.sort_index(ascending=False)  # Sort index alphabetically (A to Z)

# Plot the horizontal bar chart
fig, ax = plt.subplots(figsize=(16, 42))
bar_data_sorted.plot(
    kind="barh",
    stacked=True,
    ax=ax  # Pass the axes object for finer control
)

# Add labels and titles
ax.set_title("Duplicate Customer Names with Address Consideration by Flag", fontsize=16)
ax.set_ylabel("Customer Name", fontsize=12)
ax.set_xlabel("Count", fontsize=12)
ax.legend(title="AFF.SMC.FLG", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()


In [35]:
# Define a function to find fuzzy duplicates with additional ADDRESS1 consideration
def find_fuzzy_duplicates_with_address(df, name_col, address_col, flag_col, name_threshold, address_threshold):
    duplicates = pd.DataFrame()
    checked_pairs = set()  # Yargi - to keep track of checked (name, address) pairs

    for idx, row in df.iterrows():
        name, address = row[name_col], row[address_col]

        # Skip already checked name-address pairs
        if (name, address) in checked_pairs:
            continue

        # Find similar names and addresses
        similar_names = process.extract(name, df[name_col].unique(), scorer=fuzz.ratio, limit=None)
        similar_addresses = process.extract(address, df[address_col].unique(), scorer=fuzz.partial_ratio, limit=None)

        # Filter based on thresholds
        similar_names = [n[0] for n in similar_names if n[1] >= name_threshold]
        similar_addresses = [a[0] for a in similar_addresses if a[1] >= address_threshold]

        # Mark these name-address pairs as checked
        for sim_name in similar_names:
            for sim_address in similar_addresses:
                checked_pairs.add((sim_name, sim_address))

        # Filter rows matching similar names and addresses
        group = df[(df[name_col].isin(similar_names)) & (df[address_col].isin(similar_addresses))]

        # Check if there are different AFF.SMC.FLG values
        if group[flag_col].nunique() > 1:
            duplicates = pd.concat([duplicates, group])

    return duplicates

# Define thresholds
name_similarity_threshold = 75
address_similarity_threshold = 75 # Yargi - Lower threshold for partial matching

# Using the function on DataFrame
result_with_address = find_fuzzy_duplicates_with_address(
    duplicate_df, name_col="CUST.DESC", address_col="ADDRESS1", flag_col="AFF.SMC.FLG",
    name_threshold=name_similarity_threshold, address_threshold=address_similarity_threshold
)

# Saving the result to an Excel file
result_with_address.to_excel("fuzzy_duplicates_3.xlsx", index=False)

In [ ]:
# draw same last bar chart for the new data
# Ensure customer names are sorted alphabetically on the Y-axis
bar_data_sorted = bar_data.sort_index(ascending=False)  # Sort index alphabetically (A to Z)

# Plot the horizontal bar chart
fig, ax = plt.subplots(figsize=(16, 42))
bar_data_sorted.plot(
    kind="barh",
    stacked=True,
    ax=ax  # Pass the axes object for finer control
)

# Add labels and titles
ax.set_title("Duplicate Customer Names with Address Consideration by Flag", fontsize=16)
ax.set_ylabel("Customer Name", fontsize=12)
ax.set_xlabel("Count", fontsize=12)
ax.legend(title="AFF.SMC.FLG", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
plt.tight_layout()
plt.show()


In [1]:
import pandas as pd

# Load the two Excel files into DataFrames
df_85 = pd.read_excel("fuzzy_duplicates_2.xlsx")
df_75 = pd.read_excel("fuzzy_duplicates_3.xlsx")

# Find differences between the two DataFrames
diff_85_to_75 = pd.concat([df_85, df_75]).drop_duplicates(keep=False)

# Save the differences to a new Excel file
diff_85_to_75.to_excel("differences_between_85_and_75.xlsx", index=False)
